In [27]:
# =========================
# Imports
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

print("Imports loaded ✔")

Imports loaded ✔


In [28]:
BASE_PATH = "/kaggle/input/datasets/mouhamed221/dataset-week8/dataset/"

train_csv = pd.read_csv(BASE_PATH + "train/train.csv")
val_csv   = pd.read_csv(BASE_PATH + "val/val.csv")
test_csv  = pd.read_csv(BASE_PATH + "test/test.csv")

TRAIN_IMG = BASE_PATH + "train/images/"
TRAIN_MASK = BASE_PATH + "train/masks/"
VAL_IMG = BASE_PATH + "val/images/"
VAL_MASK = BASE_PATH + "val/masks/"
TEST_IMG = BASE_PATH + "test/images/"

print("Train:", len(train_csv), "Val:", len(val_csv), "Test:", len(test_csv))

Train: 427 Val: 90 Test: 94


## PREPROCESSING

In [29]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

val_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])
                              
print("----------------succes !---------------")

----------------succes !---------------


## DATASET and DATALOADERS

In [30]:
class SegDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["image_id"]

        # IMAGE
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        img = img.resize((256,256))
        img = np.array(img) / 255.0
        img = torch.tensor(img, dtype=torch.float32).permute(2,0,1)

        # MASK
        if self.mask_dir:
            mask_name = img_name.replace(".png", "_mask.png")
            mask = Image.open(os.path.join(self.mask_dir, mask_name)).convert("L")
            mask = mask.resize((256,256))
            mask = np.array(mask)
            mask = (mask > 127).astype(np.float32)  # 🔥 IMPORTANT BINARISATION
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

            return img, mask

        return img, img_name

# Dataloader
train_ds = SegDataset(train_csv, TRAIN_IMG, TRAIN_MASK, train_tf)
val_ds   = SegDataset(val_csv, VAL_IMG, VAL_MASK, val_tf)
test_ds  = SegDataset(test_csv, TEST_IMG, None, val_tf)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8)
test_loader  = DataLoader(test_ds, batch_size=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pos_ratio = train_csv["has_lesion"].value_counts()

pos_weight = torch.tensor([
    pos_ratio.get(0, 1) / (pos_ratio.get(1, 1) + 1e-6)
], dtype=torch.float32).to(device)
print("pos_weight:", pos_weight.item())
print("---------------SUCCES !----------------")

pos_weight: 0.0023419202771037817
---------------SUCCES !----------------


## MODEL (ResNet ENCODER + U-NET DECODER)

In [31]:
class ResNetUNet(nn.Module):
    def __init__(self):
        super().__init__()

        resnet = models.resnet34(weights="IMAGENET1K_V1")

        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.enc2 = resnet.layer1
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.pool = resnet.maxpool
        self.bottleneck = resnet.layer4

        self.up3 = nn.ConvTranspose2d(512,256,2,2)
        self.dec3 = nn.Conv2d(512,256,3,1,1)

        self.up2 = nn.ConvTranspose2d(256,128,2,2)
        self.dec2 = nn.Conv2d(256,128,3,1,1)

        self.up1 = nn.ConvTranspose2d(128,64,2,2)
        self.dec1 = nn.Conv2d(128,64,3,1,1)

        self.final = nn.Sequential(
            nn.Conv2d(64, 1, 1),
            nn.Upsample(size=(256,256), mode='bilinear', align_corners=False)
        )

    def forward(self,x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        b = self.bottleneck(e4)

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3,e4],1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2,e3],1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1,e2],1))

        return self.final(d1)


print("-----------------SUCCES !------------------")

-----------------SUCCES !------------------


## LOSS + POS_WEIGHT

In [35]:
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum()
    return 1 - (2*inter + 1e-6) / (pred.sum() + target.sum() + 1e-6)

def criterion_loss(pred, target):
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)
    return bce + dice

print("---------------SUCCES !----------------")

---------------SUCCES !----------------


## TRAINING

In [36]:
epochs = 12

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        pred = model(x)
        loss = criterion_loss(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f}")

# METRICS
def dice(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.3).float() 

    inter = (pred * target).sum()
    return (2*inter) / (pred.sum() + target.sum() + 1e-6)

# Validation
model.eval()
score = 0

with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        score += dice(model(x), y).item()

print("VAL DICE:", score/len(val_loader))
print("---------------SUCCES !----------------")

Epoch 1 | Loss: 0.3556
Epoch 2 | Loss: 0.2517
Epoch 3 | Loss: 0.1968
Epoch 4 | Loss: 0.1841
Epoch 5 | Loss: 0.1526
Epoch 6 | Loss: 0.1362
Epoch 7 | Loss: 0.1256
Epoch 8 | Loss: 0.1189
Epoch 9 | Loss: 0.1001
Epoch 10 | Loss: 0.0942
Epoch 11 | Loss: 0.0861
Epoch 12 | Loss: 0.0840
VAL DICE: 0.7601192345221838
---------------SUCCES !----------------


## SUBMISSION

In [39]:
def rle(mask):
    mask = (mask > 0.3).astype(np.uint8)
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])

    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]

    return " ".join(map(str, runs))


submission = pd.read_csv(BASE_PATH + "sample_submission.csv")

pred_dict = {}

model.eval()
with torch.no_grad():
    for x, names in test_loader:
        x = x.to(device)

        out = torch.sigmoid(model(x)).cpu().numpy()

        for m, name in zip(out, names):
            pred_dict[name] = rle(m.squeeze())


# 🔥 IMPORTANT: respect submission order
submission["rle_mask"] = submission["image_id"].map(pred_dict)

submission.to_csv("submission1.csv", index=False)

print("Submission ready ✔")

Submission ready ✔
